<a href="https://colab.research.google.com/github/RobertoSonzini/NAML_project/blob/LSMT_implementation/LSTM_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, lax
from jax import random

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import torch
from torch import nn
import math
from sklearn.preprocessing import MinMaxScaler

In [4]:
print(jax.devices())

[CudaDevice(id=0)]


In [3]:
file_path = '/content/drive/MyDrive/ProgettoNAML/all_stocks_5yr.csv'
drive.mount('/content/drive')

df = pd.read_csv(file_path)
df.head()

%matplotlib inline

from datetime import datetime

df.info()
df

Mounted at /content/drive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 619040 entries, 0 to 619039
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   date    619040 non-null  object 
 1   open    619029 non-null  float64
 2   high    619032 non-null  float64
 3   low     619032 non-null  float64
 4   close   619040 non-null  float64
 5   volume  619040 non-null  int64  
 6   Name    619040 non-null  object 
dtypes: float64(4), int64(1), object(2)
memory usage: 33.1+ MB


,date,open,high,low,close,volume,Name
0,2013-02-08,15.07,15.12,14.63,14.75,8407500,AAL
1,2013-02-11,14.89,15.01,14.26,14.46,8882000,AAL
2,2013-02-12,14.45,14.51,14.10,14.27,8126000,AAL
3,2013-02-13,14.30,14.94,14.25,14.66,10259500,AAL
4,2013-02-14,14.94,14.96,13.16,13.99,31879900,AAL
...,...,...,...,...,...,...,...
619035,2018-02-01,76.84,78.27,76.69,77.82,2982259,ZTS
619036,2018-02-02,77.53,78.12,76.73,76.78,2595187,ZTS
619037,2018-02-05,76.64,76.92,73.18,73.83,2962031,ZTS
619038,2018-02-06,72.74,74.56,72.13,73.27,4924323,ZTS


In [5]:
def sort_by_date(df):
  return df.sort_values(by='date')

In [6]:
def get_dataframe(df, company_names, moving_average=5):
    input_data_labels = ['high', 'low', 'open', 'volume', 'ma5']
    close_data = ['close']

    mask = df['Name'].isin(company_names)
    df_company = df.loc[mask].copy()

    df_company['ma5'] = df_company['close'].rolling(moving_average).mean()

    df_company = df_company.dropna()

    df_company = sort_by_date(df_company)

    return df_company[input_data_labels], df_company[close_data]

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# prendo nuovamente i dati

df_company, company_closings = get_dataframe(df, ['AAPL'])
df_company, company_closings = pd.DataFrame(scaler.fit_transform(df_company), columns=df_company.columns, index=df_company.index), pd.DataFrame(scaler.fit_transform(company_closings), columns=company_closings.columns, index=company_closings.index)

length = len(df_company)
split_idx = int(length * 0.9)

df_company_train = df_company.iloc[:split_idx]
df_company_closings_train = company_closings.iloc[:split_idx]
df_company_test = df_company.iloc[split_idx:]
df_company_closings_test = company_closings.iloc[split_idx:]

X_train = df_company_train.to_numpy().astype(np.float32)
y_train = df_company_closings_train.to_numpy().astype(np.float32)
X_test = df_company_test.to_numpy().astype(np.float32)
y_test = df_company_closings_test.to_numpy().astype(np.float32)

In [8]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, X_data, y_data, seq_len = 1):
      self.X_data = X_data  # array, tensor o lista di feature
      self.y_data = y_data  # target (opzionale se inferenza)
      self.seq_len = seq_len

    def __len__(self):
      return len(self.y_data) - self.seq_len

    def __getitem__(self, idx):
      x = self.X_data[idx : idx + self.seq_len]
      y = self.y_data[idx + self.seq_len] # il target qui lo prendo solo sul passo successivo (last step)

      # x_tensor = torch.tensor(x)
      # y_tensor = torch.tensor(y)

      return x, y

In [9]:
# per esempio di utilizzo

from torch.utils.data import DataLoader

dataset = MyDataset(X_train, y_train, seq_len = 10)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

In [10]:
def mse_loss(y_pred, y_true):
    return jnp.mean((y_pred - y_true) ** 2)

In [11]:
def sigmoid(x):
    return 1.0 / (1.0 + jnp.exp(-x))

def dsigmoid_from_output(s):
    # s = sigmoid(x)
    return s * (1 - s)

def dtanh_from_output(t):
    # t = tanh(x)
    return 1 - t ** 2

In [11]:
class LSTM_old:
  '''
    input_size: numero di feature del singolo sample
    hidden_size: numero di celle di memoria. In pratica si tratta di specificare la grandezza del layer che contiene le celle LSTM.
    output_size: numero di valori di output.
  '''
  def __init__(self, input_size, hidden_size, output_size):
    self.input_size = input_size
    self.hidden_size = hidden_size
    self.output_size = output_size

    # Output layer
    self.V = np.random.uniform(-0.1, 0.1, (output_size, hidden_size))
    self.b_out = np.zeros((output_size, 1))

    # Forget gate
    self.U_f = np.random.uniform(-0.1, 0.1, (self.hidden_size, self.input_size))
    self.W_f = np.random.uniform(-0.1, 0.1, (self.hidden_size, self.hidden_size))
    self.b_f = np.ones((hidden_size, 1)) # aiutiamo a ricordare all'inizio settando il valore a 1

    # Input gate
    self.U_i = np.random.uniform(-0.1, 0.1, (hidden_size, input_size))
    self.W_i = np.random.uniform(-0.1, 0.1, (hidden_size, hidden_size))
    self.b_i = np.zeros((hidden_size, 1))

    # External input gate (Candidate)
    self.U_g = np.random.uniform(-0.1, 0.1, (hidden_size, input_size))
    self.W_g = np.random.uniform(-0.1, 0.1, (hidden_size, hidden_size))
    self.b_g = np.zeros((hidden_size, 1))

    # Output gate
    self.U_o = np.random.uniform(-0.1, 0.1, (hidden_size, input_size))
    self.W_o = np.random.uniform(-0.1, 0.1, (hidden_size, hidden_size))
    self.b_o = np.zeros((hidden_size, 1))

    # Initial states
    self.h = np.zeros((hidden_size, 1))
    self.s = np.zeros((hidden_size, 1))

  def forward_step(self, x_t):
    i_t = sigmoid(np.dot(self.U_i, x_t) + np.dot(self.W_i, self.h) + self.b_i)
    f_t = sigmoid(np.dot(self.U_f, x_t) + np.dot(self.W_f, self.h) + self.b_f)
    g_t = np.tanh(np.dot(self.U_g, x_t) + np.dot(self.W_g, self.h) + self.b_g)
    o_t = sigmoid(np.dot(self.U_o, x_t) + np.dot(self.W_o, self.h) + self.b_o)

    self.s = f_t * self.s + i_t * g_t
    self.h = o_t * np.tanh(self.s)

    return self.h, self.s

  def forward(self, X):
    """
    X: array di dimensione (seq_len, input_size)
    Ritorna:
        Y_seq: array di dimensione (seq_len, output_size)
    """
    Y_seq = []

    for t in range(X.shape[0]):
        x_t = X[t].reshape(-1, 1)
        h_t, _ = self.forward_step(x_t)

        # Output layer
        y_t = np.dot(self.V, h_t) + self.b_out
        Y_seq.append(y_t)

    # Converti in array (seq_len, output_size)
    Y_seq = np.hstack(Y_seq).T

    return Y_seq



In [12]:
class LSTM_not_optimized:
    def __init__(self, key, input_size, hidden_size, output_size, loss_function):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.loss_function = loss_function
        self.key = key

        # Split the key for different initializations
        keys = jax.random.split(self.key, 9)

        # Inizializzazione pesi
        self.params = {
            'U_f': jax.random.uniform(keys[0], (hidden_size, input_size), minval=-0.1, maxval=0.1),
            'W_f': jax.random.uniform(keys[1], (hidden_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_f': jnp.ones((hidden_size, 1)),

            'U_i': jax.random.uniform(keys[2], (hidden_size, input_size), minval=-0.1, maxval=0.1),
            'W_i': jax.random.uniform(keys[3], (hidden_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_i': jnp.zeros((hidden_size, 1)),

            'U_g': jax.random.uniform(keys[4], (hidden_size, input_size), minval=-0.1, maxval=0.1),
            'W_g': jax.random.uniform(keys[5], (hidden_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_g': jnp.zeros((hidden_size, 1)),

            'U_o': jax.random.uniform(keys[6], (hidden_size, input_size), minval=-0.1, maxval=0.1),
            'W_o': jax.random.uniform(keys[7], (hidden_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_o': jnp.zeros((hidden_size, 1)),

            'V': jax.random.uniform(keys[8], (output_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_out': jnp.zeros((output_size, 1)),
        }

    def forward_step(self, params, x_t, h_prev, s_prev):
        i_t = sigmoid(params['U_i'] @ x_t + params['W_i'] @ h_prev + params['b_i'])
        f_t = sigmoid(params['U_f'] @ x_t + params['W_f'] @ h_prev + params['b_f'])
        g_t = jnp.tanh(params['U_g'] @ x_t + params['W_g'] @ h_prev + params['b_g'])
        o_t = sigmoid(params['U_o'] @ x_t + params['W_o'] @ h_prev + params['b_o'])

        s_t = f_t * s_prev + i_t * g_t
        h_t = o_t * jnp.tanh(s_t)

        y_t = params['V'] @ h_t + params['b_out']
        return h_t, s_t, y_t

    def forward(self, params, X):
        h = jnp.zeros((self.hidden_size, 1))
        s = jnp.zeros((self.hidden_size, 1))
        Y_seq = []

        for t in range(X.shape[0]):
            x_t = X[t].reshape(-1, 1)
            h, s, y_t = self.forward_step(params, x_t, h, s)
            Y_seq.append(y_t)

        return jnp.hstack(Y_seq).T  # (seq_len, output_size)

    def loss(self, params, X, Y_true):
        Y_pred = self.forward(params, X)
        return self.loss_function(Y_pred, Y_true)

    def backward(self, X, Y_true, lr=0.01):
        # JAX calcola automaticamente i gradienti
        grads = jax.grad(self.loss)(self.params, X, Y_true)
        # Aggiorna pesi
        for key in self.params.keys():
            self.params[key] = self.params[key] - lr * grads[key]

In [43]:
class LSTM:
    def __init__(self, key, input_size, hidden_size, output_size, loss_function):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.loss_function = loss_function
        self.key = key

        # Split the key for different initializations
        keys = jax.random.split(self.key, 9)

        # Inizializzazione pesi
        self.params = {
            'U_f': jax.random.uniform(keys[0], (hidden_size, input_size), minval=-0.1, maxval=0.1),
            'W_f': jax.random.uniform(keys[1], (hidden_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_f': jnp.ones((hidden_size, )),

            'U_i': jax.random.uniform(keys[2], (hidden_size, input_size), minval=-0.1, maxval=0.1),
            'W_i': jax.random.uniform(keys[3], (hidden_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_i': jnp.zeros((hidden_size, )),

            'U_g': jax.random.uniform(keys[4], (hidden_size, input_size), minval=-0.1, maxval=0.1),
            'W_g': jax.random.uniform(keys[5], (hidden_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_g': jnp.zeros((hidden_size, )),

            'U_o': jax.random.uniform(keys[6], (hidden_size, input_size), minval=-0.1, maxval=0.1),
            'W_o': jax.random.uniform(keys[7], (hidden_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_o': jnp.zeros((hidden_size, )),

            'V': jax.random.uniform(keys[8], (output_size, hidden_size), minval=-0.1, maxval=0.1),
            'b_out': jnp.zeros((output_size, )),
        }

    def forward_step(self, params, x_t, h_prev, s_prev):
        i_t = sigmoid(params['U_i'] @ x_t + params['W_i'] @ h_prev + params['b_i'])
        f_t = sigmoid(params['U_f'] @ x_t + params['W_f'] @ h_prev + params['b_f'])
        g_t = jnp.tanh(params['U_g'] @ x_t + params['W_g'] @ h_prev + params['b_g'])
        o_t = sigmoid(params['U_o'] @ x_t + params['W_o'] @ h_prev + params['b_o'])

        s_t = f_t * s_prev + i_t * g_t
        h_t = o_t * jnp.tanh(s_t)

        y_t = params['V'] @ h_t + params['b_out']
        return h_t, s_t, y_t

    def forward(self, params, X):
      # X: (seq_len, input_size)
      h0 = jnp.zeros((self.hidden_size, ))
      s0 = jnp.zeros((self.hidden_size, ))

      def step_scan(carry, x_t):
        h_prev, s_prev = carry
        h_t, s_t, y_t = self.forward_step(params, x_t, h_prev, s_prev)
        return (h_t, s_t), y_t

      (h_final, s_final), Y_seq = lax.scan(step_scan, (h0, s0), X)
      return jnp.stack(Y_seq, axis=0)  # (seq_len, output_size)

    def forward_batch(self, params, X_batch):
      # X_batch: (batch_size, seq_len, input_size)
      return vmap(lambda X_seq: self.forward(params, X_seq))(X_batch)

    def loss(self, params, X, Y_true):
        Y_pred = self.forward(params, X)
        return self.loss_function(Y_pred, Y_true)

    def backward_batch(self, params, X_batch, Y_batch, lr=0.01):
      # Funzione pura per il grad
      def batch_loss(p):
        return jnp.mean(vmap(lambda X_seq, Y_seq: self.loss(p, X_seq, Y_seq))
                        (X_batch, Y_batch))

      grads = jax.grad(batch_loss)(params)

      # Aggiorna i parametri in modo puro
      new_params = {k: params[k] - lr * grads[k] for k in params.keys()}
      return new_params

In [46]:
# hyperparameters
input_size = 5
hidden_size = 32
num_epochs = 500
lr = 0.001
seq_len = 40
output_size = 1

key = random.PRNGKey(0)
model = LSTM(key, input_size, hidden_size, output_size, loss_function=mse_loss)

In [51]:
def train_LSTM(model, X_train, y_train, seq_len, batch_size=32, learning_rate=0.001, epochs=500):
    """
    Train the LSTM using a PyTorch-style DataLoader interno.
    X_train, y_train: dati di training
    seq_len: lunghezza delle sequenze
    batch_size: dimensione del batch
    learning_rate: learning rate
    epochs: numero di epoche
    """
    # Crea dataset e loader
    dataset = MyDataset(X_train, y_train, seq_len=seq_len)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    losses = []

    for epoch in range(epochs):
        epoch_loss = 0.0

        for i, (samples, labels) in enumerate(loader):
            # Converti da PyTorch Tensor a JAX numpy array
            # Ensure samples and labels are float32 for compatibility with JAX
            X_batch = jnp.array(samples, dtype=jnp.float32)
            y_batch = jnp.array(labels, dtype=jnp.float32)

            # questo perchè per ora opero senza batches
            # X_batch = jnp.squeeze(X_batch, axis = 0)
            # y_batch = jnp.squeeze(y_batch, axis = 0)

            model.params = model.backward_batch(model.params, X_batch, y_batch, lr=learning_rate)

            # Calcola la loss del batch per monitoraggio
            # Calcola la loss del batch per monitoraggio
            batch_loss = jnp.mean(
                jax.vmap(lambda X_seq, Y_seq: model.loss(model.params, X_seq, Y_seq))(
                    X_batch, y_batch
                )
            )
            epoch_loss += batch_loss

        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)

        print(f'Epoch [{epoch}/{epochs}], Loss: {avg_loss:.4f} [{datetime.now()}]')

    return losses

In [53]:
import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, lax

# Esempio di funzione sigmoid
def sigmoid(x):
    return 1 / (1 + jnp.exp(-x))

# Batch loss JIT-ed
@jit
def batch_loss(params, X_batch, Y_batch, model):
    return jnp.mean(
        vmap(lambda X_seq, Y_seq: model.loss(params, X_seq, Y_seq))(X_batch, Y_batch)
    )

# Aggiornamento parametri JIT-ed
@jit
def update(params, X_batch, Y_batch, lr, model):
    grads = grad(batch_loss)(params, X_batch, Y_batch, model)
    new_params = {k: params[k] - lr * grads[k] for k in params.keys()}
    return new_params

# Funzione di training
def train_LSTM_jit(model, X_train, y_train, seq_len, batch_size=32, learning_rate=0.001, epochs=500):
    dataset = MyDataset(X_train, y_train, seq_len=seq_len)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    losses = []

    for epoch in range(epochs):
        epoch_loss = 0.0
        for i, (samples, labels) in enumerate(loader):
            X_batch = jnp.array(samples, dtype=jnp.float32)
            Y_batch = jnp.array(labels, dtype=jnp.float32)

            # Aggiorna parametri con JIT
            model.params = update(model.params, X_batch, Y_batch, learning_rate, model)

            # Loss batch (JIT)
            batch_l = batch_loss(model.params, X_batch, Y_batch, model)
            epoch_loss += batch_l

        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
        if (epoch + 1) % 50 == 0:
            print(f'Epoch [{epoch}/{epochs}], Loss: {avg_loss:.4f}')

    return losses


In [54]:
losses = train_LSTM_jit(model, X_train, y_train, seq_len=seq_len, learning_rate=lr, epochs=num_epochs)

TypeError: Error interpreting argument to <function update at 0x79ce4855e840> as an abstract array. The problematic value is of type <class '__main__.LSTM'> and was passed to the function at path model.
This typically means that a jit-wrapped function was called with a non-array argument, and this argument was not marked as static using the static_argnums or static_argnames parameters of jax.jit.